# Chapter 21: Implicit Modeling

**Source span.** This notebook is grounded in *Fundamentals of Computer Graphics*, Chapter 21, printed pages 595-622 and physical PDF pages 612-639. The source span introduces implicit scalar fields, skeletal primitives, summation and Ricci-style blending, gradients and normals, distance fields, level sets, variational and convolution surfaces, rendering by ray search or polygonization, cubic space partitioning, polygonization ambiguity, CSG, warping, BlobTrees, and interactive implicit-modeling systems.

**Chapter goal.** Learn to design, inspect, and validate an implicit model as a scalar field: a function that can be sampled, differentiated, composed, warped, polygonized, and cached. The notebook keeps the models synthetic and small so each idea can be checked numerically without requiring a production renderer.

An implicit surface is not stored as vertices first. It is the set of points where a field reaches a chosen level. That shift explains both the power and the cost of the chapter: blends and booleans are simple field expressions, but rendering needs root finding, spatial search, or extraction to a polygon mesh.


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
from matplotlib import patches
from matplotlib.collections import LineCollection
import networkx as nx
import numpy as np
import plotly.graph_objects as go
import sympy as sp
import trimesh
from skimage import measure

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the FCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import (
    assert_artifacts,
    artifact_dir,
    display_artifact,
    save_json,
    save_matplotlib,
    save_plotly_html,
    save_table_csv,
)
from utils.notebook_checks import assert_nonblank_image
from utils.plotting import PALETTE, close, style_axis

CHAPTER = 21
TOPIC = "chapter-21"
TITLE = "Implicit Modeling"
PRINTED_PAGES = "595-622"
PDF_PAGES = "612-639"
ISO = 0.45

artifact_dir(TOPIC, "figures")
artifact_dir(TOPIC, "html")
artifact_dir(TOPIC, "checks")
artifact_dir(TOPIC, "tables")

generated_artifacts: list[Path] = []
check_artifacts: list[Path] = []
validation_data: dict[str, object] = {}


## Translation guide

| Source concept | Computational object used here | What to inspect |
| --- | --- | --- |
| Implicit function | A scalar function `F(p)` sampled on a grid | A level contour separates field values above and below the chosen iso value. |
| Skeletal primitive | Distance to a point, segment, circle, or box passed through a compact falloff | The skeleton is lower-dimensional; the visible solid is an iso set around it. |
| Summation blend | `F = F1 + F2 + ...` | Fields add before the surface is extracted, so nearby pieces bulge and merge. |
| Smooth normal | `grad F / ||grad F||` | Normal direction comes from the field, not from a mesh vertex list. |
| Space partitioning | Cubes or squares whose corner signs bracket the iso value | Extraction spends work only where a cell may contain the surface. |
| CSG and Ricci/R-style operators | `max`, `min`, `min(A, -B)`, and smooth power blends | Boolean composition is algebra on fields, with sharp operators creating creases. |
| Warping | Evaluate a base field at inverse-warped sample coordinates | The surface moves because the query point is moved before distance is measured. |
| BlobTree | An expression tree of primitives, blends, booleans, transforms, and warps | Traversal evaluates a field value and gradient by composing node operations. |

## Visual storyboard

1. `implicit-scalar-field-level-set.png`: a scalar field, an iso contour, and gradient normals.
2. `skeletal-primitives-distance-fields.png`: point, line-segment, and ring skeletons converted to fields.
3. `summation-ricci-blending-family.png`: additive blending compared with Ricci-style blends and hard union.
4. `gradient-normal-rendering-cues.png`: analytic and numerical gradients as rendering normals.
5. `space-partitioning-marching-cells.png`: grid cells, edge crossings, and an ambiguous marching case.
6. `csg-operators-and-creases.png`: union, intersection, and difference as field operators.
7. `warp-inverse-sampling.png`: twist, taper, and bend as inverse sampling maps.
8. `blobtree-evaluation-graph.png`: a BlobTree expression and its evaluation order.
9. `marching-cubes-synthetic-implicit-field.html`: a synthetic 3D field extracted with marching cubes.

Each visual has a check: a size/nonblank artifact check plus a field, gradient, topology, or traversal invariant.


In [ ]:
def record_artifact(path: Path) -> Path:
    generated_artifacts.append(Path(path))
    return Path(path)


def record_check(path: Path) -> Path:
    check_artifacts.append(Path(path))
    return Path(path)


def save_figure(fig, filename: str, *, dpi: int = 170) -> Path:
    path = save_matplotlib(fig, TOPIC, filename, dpi=dpi)
    close(fig)
    return record_artifact(path)


def save_check(data: object, filename: str) -> Path:
    return record_check(save_json(data, TOPIC, filename))


def compact_falloff(distance, support: float = 1.0):
    # Compact C2-style falloff: one at the skeleton, zero at the support radius.
    q = np.asarray(distance, dtype=float) / support
    return np.where(q < 1.0, (1.0 - q * q) ** 3, 0.0)


def xy_grid(xlim=(-2.25, 2.25), ylim=(-1.75, 1.75), n=340):
    x = np.linspace(*xlim, n)
    y = np.linspace(*ylim, n)
    X, Y = np.meshgrid(x, y, indexing="xy")
    P = np.stack([X, Y], axis=-1)
    return x, y, X, Y, P


def segment_distance_2d(points, a, b):
    p = np.asarray(points, dtype=float)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    ab = b - a
    denom = float(np.dot(ab, ab))
    t = np.clip(np.sum((p - a) * ab, axis=-1) / denom, 0.0, 1.0)
    closest = a + t[..., None] * ab
    return np.linalg.norm(p - closest, axis=-1), closest, t


def segment_distance_3d(points, a, b):
    p = np.asarray(points, dtype=float)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    ab = b - a
    denom = float(np.dot(ab, ab))
    t = np.clip(np.sum((p - a) * ab, axis=-1) / denom, 0.0, 1.0)
    closest = a + t[..., None] * ab
    return np.linalg.norm(p - closest, axis=-1), closest, t


def positive_circle_field(points, center=(0.0, 0.0), radius=1.0):
    p = np.asarray(points, dtype=float)
    return radius - np.linalg.norm(p - np.asarray(center, dtype=float), axis=-1)


def positive_box_field(points, center=(0.0, 0.0), half=(0.7, 0.45)):
    p = np.asarray(points, dtype=float) - np.asarray(center, dtype=float)
    h = np.asarray(half, dtype=float)
    q = np.abs(p) - h
    outside = np.linalg.norm(np.maximum(q, 0.0), axis=-1)
    inside = np.minimum(np.maximum(q[..., 0], q[..., 1]), 0.0)
    signed_distance_negative_inside = outside + inside
    return -signed_distance_negative_inside


def finite_difference_gradient(field_fn, x, y, h=1e-4):
    p_xp = np.stack([x + h, y], axis=-1)
    p_xm = np.stack([x - h, y], axis=-1)
    p_yp = np.stack([x, y + h], axis=-1)
    p_ym = np.stack([x, y - h], axis=-1)
    gx = (field_fn(p_xp) - field_fn(p_xm)) / (2 * h)
    gy = (field_fn(p_yp) - field_fn(p_ym)) / (2 * h)
    return gx, gy


## 1. Scalar fields and level sets

A field stores more than the final surface. It tells us which side of the surface a query point is on, how far a simple model thinks the point is from the shape, and which direction the surface normal should point. In this notebook, positive values usually mean inside and the surface is the zero set, except for compact skeletal fields where we explicitly use a positive iso value.

The first figure uses a signed circle field `F(x, y) = r - sqrt(x^2 + y^2)`. The zero contour is the model. The arrows are normalized gradients sampled near that contour. A ray tracer would solve `F(ray(t)) = 0`; a polygonizer would find grid cells whose corner values straddle zero.


In [ ]:
x, y, X, Y, P = xy_grid(n=320)
R = 1.05
circle_field = positive_circle_field(P, radius=R)

sample_theta = np.linspace(0, 2 * np.pi, 28, endpoint=False)
px = R * np.cos(sample_theta)
py = R * np.sin(sample_theta)
gx = -px / R
gy = -py / R
outward_x = -gx
outward_y = -gy

fig, ax = plt.subplots(figsize=(8.4, 6.3))
levels = np.linspace(-1.15, 1.05, 18)
cf = ax.contourf(X, Y, circle_field, levels=levels, cmap="RdYlBu_r", alpha=0.86)
ax.contour(X, Y, circle_field, levels=[0.0], colors="black", linewidths=2.4)
ax.quiver(px, py, outward_x, outward_y, color="black", scale=24, width=0.004, label="outward normal")
ax.scatter([0], [0], s=36, color=PALETTE["red"], zorder=5, label="skeleton point")
fig.colorbar(cf, ax=ax, shrink=0.86, label="field value F(x,y)")
style_axis(ax, "A surface is an iso set of a scalar field", equal=True, xlabel="x", ylabel="y")
ax.legend(loc="upper right")
ax.text(-2.15, -1.56, "inside: F > 0   surface: F = 0   outside: F < 0", fontsize=9, color=PALETTE["ink"])
scalar_field_path = save_figure(fig, "implicit-scalar-field-level-set.png")

grad_norms = np.sqrt(gx * gx + gy * gy)
validation_data["scalar_field"] = {
    "center_value": float(positive_circle_field(np.array([[0.0, 0.0]]), radius=R)[0]),
    "boundary_max_abs": float(np.max(np.abs(positive_circle_field(np.stack([px, py], axis=-1), radius=R)))),
    "normal_norm_min": float(np.min(grad_norms)),
    "normal_norm_max": float(np.max(grad_norms)),
}
display_artifact(scalar_field_path, width=760)
validation_data["scalar_field"]


## 2. Skeletal primitives and falloff filters

A skeletal primitive separates the geometry of the skeleton from the filter that turns distance into field strength. A point skeleton gives round blobs. A line segment gives a capsule-like influence region. A ring skeleton gives a tube-like influence in 2D cross-section. The same rendering code can sample all three because each primitive answers the same question: what is the distance from this point to the skeleton?

The compact falloff used here is intentionally simple: it is one at the skeleton and becomes exactly zero at a finite support radius. That finite support is useful for acceleration because faraway primitives do not contribute.

The chapter also points to distance-field and F-rep terminology, variational implicit surfaces, and convolution surfaces. Those are different ways to define the same kind of object for this notebook: a procedure that returns a scalar field value and, when rendering quality matters, a reliable gradient. The examples stay with compact skeletal fields so the later blending, CSG, warping, and marching-cubes checks remain small enough to audit.


In [ ]:
x, y, X, Y, P = xy_grid(xlim=(-2.2, 2.2), ylim=(-1.45, 1.45), n=300)
point_center = np.array([-0.75, 0.0])
point_distance = np.linalg.norm(P - point_center, axis=-1)
point_field = compact_falloff(point_distance, support=0.95)

seg_a = np.array([-1.35, -0.35])
seg_b = np.array([1.2, 0.45])
segment_distance, closest, t = segment_distance_2d(P, seg_a, seg_b)
segment_field = compact_falloff(segment_distance, support=0.55)

ring_center = np.array([0.15, -0.02])
ring_radius = 0.78
ring_distance = np.abs(np.linalg.norm(P - ring_center, axis=-1) - ring_radius)
ring_field = compact_falloff(ring_distance, support=0.28)

fig, axes = plt.subplots(1, 3, figsize=(13.6, 4.25), constrained_layout=True)
for ax, field, title in [
    (axes[0], point_field, "point skeleton"),
    (axes[1], segment_field, "line-segment skeleton"),
    (axes[2], ring_field, "circle skeleton"),
]:
    im = ax.imshow(field, extent=[x.min(), x.max(), y.min(), y.max()], origin="lower", cmap="magma", vmin=0, vmax=1)
    ax.contour(X, Y, field, levels=[ISO], colors="white", linewidths=2)
    style_axis(ax, title, equal=True, xlabel="x", ylabel="y")
axes[0].scatter([point_center[0]], [point_center[1]], color="cyan", edgecolor="black", zorder=4)
axes[1].plot([seg_a[0], seg_b[0]], [seg_a[1], seg_b[1]], color="cyan", linewidth=4, solid_capstyle="round")
axes[2].add_patch(patches.Circle(ring_center, ring_radius, fill=False, color="cyan", linewidth=3))
fig.colorbar(im, ax=axes, shrink=0.82, label="falloff field")
skeletal_path = save_figure(fig, "skeletal-primitives-distance-fields.png")

validation_data["skeletal_primitives"] = {
    "point_max": float(point_field.max()),
    "segment_max": float(segment_field.max()),
    "ring_max": float(ring_field.max()),
    "finite_support_fraction_point": float(np.mean(point_field == 0.0)),
    "segment_closest_parameter_range": [float(t.min()), float(t.max())],
}
display_artifact(skeletal_path, width=880)
validation_data["skeletal_primitives"]


## 3. Blending: summation, Ricci power blends, and hard union

When fields are added, the surface is extracted after the addition. That makes nearby primitives influence each other before polygonization or ray tracing sees any vertices. The same two source fields can also be combined with a Ricci-style power blend. As the exponent grows, the operation moves toward the hard `max` union, reducing the soft bulge between primitives.

This section uses two compact point fields. The visible contour is `F = iso`, not `F = 0`, because these blobby fields are nonnegative potentials.


In [ ]:
def two_point_fields(P, separation=1.15, support=1.0):
    c0 = np.array([-separation / 2.0, 0.0])
    c1 = np.array([separation / 2.0, 0.0])
    f0 = compact_falloff(np.linalg.norm(P - c0, axis=-1), support=support)
    f1 = compact_falloff(np.linalg.norm(P - c1, axis=-1), support=support)
    return f0, f1, c0, c1


def ricci_blend(fields, exponent):
    stack = np.stack([np.maximum(f, 0.0) for f in fields], axis=0)
    if math.isinf(exponent):
        return np.max(stack, axis=0)
    return np.sum(stack ** exponent, axis=0) ** (1.0 / exponent)


x, y, X, Y, P = xy_grid(xlim=(-2.0, 2.0), ylim=(-1.35, 1.35), n=320)
f0, f1, c0, c1 = two_point_fields(P, separation=1.08, support=1.02)
fields = [f0, f1]
blend_cases = [
    ("sum: F0 + F1", f0 + f1),
    ("Ricci n=2", ricci_blend(fields, 2.0)),
    ("Ricci n=8", ricci_blend(fields, 8.0)),
    ("hard union: max", ricci_blend(fields, math.inf)),
]

fig, axes = plt.subplots(1, 4, figsize=(15.2, 3.85), constrained_layout=True)
for ax, (title, field) in zip(axes, blend_cases):
    ax.contourf(X, Y, field, levels=np.linspace(0, max(1.0, float(field.max())), 16), cmap="viridis")
    ax.contour(X, Y, field, levels=[ISO], colors="white", linewidths=2.2)
    ax.scatter([c0[0], c1[0]], [c0[1], c1[1]], color="black", s=22)
    style_axis(ax, title, equal=True, xlabel="x", ylabel="y")
    ax.set_xlim(-1.75, 1.75)
    ax.set_ylim(-1.18, 1.18)
blend_path = save_figure(fig, "summation-ricci-blending-family.png")

center_index = (np.abs(y).argmin(), np.abs(x).argmin())
validation_data["blending"] = {
    "iso": ISO,
    "sum_center_value": float((f0 + f1)[center_index]),
    "union_center_value": float(np.maximum(f0, f1)[center_index]),
    "sum_greater_or_equal_operands": bool(np.all(f0 + f1 >= f0) and np.all(f0 + f1 >= f1)),
    "hard_union_equals_max": bool(np.allclose(ricci_blend(fields, math.inf), np.maximum(f0, f1))),
}
display_artifact(blend_path, width=920)
validation_data["blending"]


## 4. Gradients, normals, and rendering cues

Implicit rendering depends on field evaluation and normal evaluation. For a signed field, the normal is a normalized gradient, possibly with a sign flip depending on the inside convention. If the field is built from smooth falloff filters and smooth operators, the normal changes smoothly. If a CSG operator chooses one branch abruptly, the normal can jump at a crease.

The figure compares an analytic gradient for a circle with a finite-difference gradient for a blended field. It also adds a simple diffuse shading cue: the dot product between the outward normal and a light direction. This is the minimum data a ray tracer needs once it has found the surface point.


In [ ]:
x, y, X, Y, P = xy_grid(xlim=(-1.8, 1.8), ylim=(-1.45, 1.45), n=300)
f0, f1, c0, c1 = two_point_fields(P, separation=1.0, support=1.05)
blend_field = f0 + f1 - ISO


def blend_level_field(points):
    f0_local, f1_local, *_ = two_point_fields(np.asarray(points), separation=1.0, support=1.05)
    return f0_local + f1_local - ISO


sample_contours = measure.find_contours(blend_field, 0.0)
main_contour = max(sample_contours, key=len)
step = max(len(main_contour) // 34, 1)
contour_samples = main_contour[::step]
sample_y = y[0] + (y[-1] - y[0]) * contour_samples[:, 0] / (len(y) - 1)
sample_x = x[0] + (x[-1] - x[0]) * contour_samples[:, 1] / (len(x) - 1)
gx_num, gy_num = finite_difference_gradient(blend_level_field, sample_x, sample_y)
lengths = np.sqrt(gx_num * gx_num + gy_num * gy_num)
normal_x = gx_num / np.maximum(lengths, 1e-12)
normal_y = gy_num / np.maximum(lengths, 1e-12)
light = np.array([0.35, 0.94])
light = light / np.linalg.norm(light)
shade = np.clip(normal_x * light[0] + normal_y * light[1], 0.0, 1.0)

x_sym, y_sym, r_sym = sp.symbols("x y r", positive=True)
F_sym = r_sym**2 - x_sym**2 - y_sym**2
sym_grad = [sp.simplify(sp.diff(F_sym, x_sym)), sp.simplify(sp.diff(F_sym, y_sym))]

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.2), constrained_layout=True)
axes[0].contourf(X, Y, positive_circle_field(P, radius=R), levels=18, cmap="RdYlBu_r", alpha=0.82)
axes[0].contour(X, Y, positive_circle_field(P, radius=R), levels=[0.0], colors="black", linewidths=2)
axes[0].quiver(px, py, outward_x, outward_y, color="black", scale=22, width=0.004)
style_axis(axes[0], "analytic circle normals", equal=True, xlabel="x", ylabel="y")
axes[0].text(-1.68, -1.27, "SymPy gradient of r^2 - x^2 - y^2: " + str(sym_grad), fontsize=8)

axes[1].contourf(X, Y, blend_level_field(P), levels=np.linspace(-0.45, 1.1, 18), cmap="viridis", alpha=0.88)
axes[1].contour(X, Y, blend_level_field(P), levels=[0.0], colors="white", linewidths=2)
q = axes[1].quiver(sample_x, sample_y, normal_x, normal_y, shade, cmap="plasma", scale=22, width=0.0045)
axes[1].scatter([c0[0], c1[0]], [c0[1], c1[1]], color="black", s=24)
style_axis(axes[1], "finite-difference normals on a blend", equal=True, xlabel="x", ylabel="y")
fig.colorbar(q, ax=axes[1], shrink=0.78, label="Lambert term")
gradient_path = save_figure(fig, "gradient-normal-rendering-cues.png")

validation_data["gradients"] = {
    "symbolic_gradient": [str(item) for item in sym_grad],
    "blend_normal_length_mean": float(np.mean(np.sqrt(normal_x**2 + normal_y**2))),
    "blend_normal_length_max_error": float(np.max(np.abs(np.sqrt(normal_x**2 + normal_y**2) - 1.0))),
    "shade_min_max": [float(shade.min()), float(shade.max())],
}
display_artifact(gradient_path, width=860)
validation_data["gradients"]


## 5. Space partitioning and marching cells

A polygonizer does not know the surface location until it samples. In a cubic grid, a cell is interesting when its corner values lie on both sides of the iso value. The same idea appears in 2D as marching squares and in 3D as marching cubes. The cell address is a bit pattern of inside/outside corner tests; a lookup table converts that address into crossed edges and triangles.

The first panel below marks every grid square whose corners bracket the surface. The second panel draws the line segments produced by linear interpolation along crossed edges. The third panel is a deliberately ambiguous marching-squares case: opposite corners are inside. Extra samples or tetrahedral/cell subdivision are common ways to break this ambiguity.


In [ ]:
def marching_square_segments(x, y, F, level=0.0):
    segments = []
    active_cells = []
    ambiguous = []
    for j in range(len(y) - 1):
        for i in range(len(x) - 1):
            corners = [
                (x[i], y[j], F[j, i]),
                (x[i + 1], y[j], F[j, i + 1]),
                (x[i + 1], y[j + 1], F[j + 1, i + 1]),
                (x[i], y[j + 1], F[j + 1, i]),
            ]
            inside = [value >= level for _, _, value in corners]
            if all(inside) or not any(inside):
                continue
            active_cells.append((i, j))
            if sum(inside) == 2 and inside[0] == inside[2]:
                ambiguous.append((i, j))
            if sum(inside) == 2 and inside[1] == inside[3]:
                ambiguous.append((i, j))
            crossings = []
            for a, b in [(0, 1), (1, 2), (2, 3), (3, 0)]:
                x0, y0, v0 = corners[a]
                x1, y1, v1 = corners[b]
                if (v0 - level) == 0 and (v1 - level) == 0:
                    continue
                if (v0 >= level) != (v1 >= level):
                    alpha = (level - v0) / (v1 - v0)
                    crossings.append((x0 + alpha * (x1 - x0), y0 + alpha * (y1 - y0)))
            if len(crossings) == 2:
                segments.append([crossings[0], crossings[1]])
            elif len(crossings) == 4:
                segments.append([crossings[0], crossings[1]])
                segments.append([crossings[2], crossings[3]])
    return segments, active_cells, ambiguous


coarse_x = np.linspace(-1.65, 1.65, 23)
coarse_y = np.linspace(-1.25, 1.25, 19)
CX, CY = np.meshgrid(coarse_x, coarse_y, indexing="xy")
CP = np.stack([CX, CY], axis=-1)
coarse_field = positive_circle_field(CP, radius=0.93) + 0.22 * np.cos(3 * CX) * np.sin(2 * CY)
segments, active_cells, ambiguous_cells = marching_square_segments(coarse_x, coarse_y, coarse_field, level=0.0)

amb_values = np.array([[1.0, -1.0], [-1.0, 1.0]])

fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.35), constrained_layout=True)
axes[0].contourf(CX, CY, coarse_field, levels=16, cmap="RdYlBu_r", alpha=0.75)
axes[0].contour(CX, CY, coarse_field, levels=[0.0], colors="black", linewidths=2)
for i, j in active_cells:
    rect = patches.Rectangle((coarse_x[i], coarse_y[j]), coarse_x[i + 1] - coarse_x[i], coarse_y[j + 1] - coarse_y[j], fill=False, edgecolor=PALETTE["gold"], linewidth=1.2)
    axes[0].add_patch(rect)
axes[0].set_xticks(coarse_x[::3])
axes[0].set_yticks(coarse_y[::3])
style_axis(axes[0], "cells that bracket the surface", equal=True, xlabel="x", ylabel="y")

axes[1].contourf(CX, CY, coarse_field, levels=16, cmap="Greys", alpha=0.28)
axes[1].add_collection(LineCollection(segments, colors=PALETTE["red"], linewidths=2.2))
axes[1].scatter(CX, CY, s=9, color=PALETTE["ink"], alpha=0.65)
axes[1].set_xlim(coarse_x.min(), coarse_x.max())
axes[1].set_ylim(coarse_y.min(), coarse_y.max())
style_axis(axes[1], "linear edge interpolation", equal=True, xlabel="x", ylabel="y")

axes[2].imshow(amb_values, extent=[0, 1, 0, 1], origin="lower", cmap="RdYlBu_r", vmin=-1, vmax=1, alpha=0.8)
for xx in [0, 1]:
    axes[2].plot([xx, xx], [0, 1], color="black", linewidth=1)
for yy in [0, 1]:
    axes[2].plot([0, 1], [yy, yy], color="black", linewidth=1)
axes[2].scatter([0, 1], [0, 1], s=95, color=PALETTE["blue"], label="inside")
axes[2].scatter([1, 0], [0, 1], s=95, color=PALETTE["red"], label="outside")
axes[2].plot([0.5, 0.5], [0, 1], color="white", linewidth=2.5, label="one valid connection")
axes[2].plot([0, 1], [0.5, 0.5], color="black", linewidth=2.5, linestyle="--", label="another valid connection")
axes[2].set_xlim(-0.1, 1.1)
axes[2].set_ylim(-0.1, 1.1)
style_axis(axes[2], "ambiguous cell", equal=True, xlabel="local x", ylabel="local y")
axes[2].legend(fontsize=8, loc="upper center")
partition_path = save_figure(fig, "space-partitioning-marching-cells.png")

validation_data["partitioning"] = {
    "active_cell_count": len(active_cells),
    "marching_segment_count": len(segments),
    "ambiguous_cell_count": len(ambiguous_cells),
    "all_segments_have_two_endpoints": bool(all(len(segment) == 2 for segment in segments)),
}
display_artifact(partition_path, width=900)
validation_data["partitioning"]


## 6. CSG as field algebra

Constructive solid geometry becomes simple when each object is already a field. With the positive-inside convention used here, union is `max(A, B)`, intersection is `min(A, B)`, and difference is `min(A, -B)`. The algebra is compact, but it is not automatically smooth. Where the selected branch changes, the gradient can jump; that is the crease visible in sharp CSG joins.

The next figure uses a circle and a box because their disagreement is easy to inspect. The dashed line marks the branch equality locus where a `min` or `max` operator may switch which primitive supplies the result.


In [ ]:
x, y, X, Y, P = xy_grid(xlim=(-1.8, 1.8), ylim=(-1.35, 1.35), n=330)
A = positive_circle_field(P, center=(-0.35, 0.0), radius=0.92)
B = positive_box_field(P, center=(0.35, 0.02), half=(0.78, 0.48))
union = np.maximum(A, B)
intersection = np.minimum(A, B)
difference = np.minimum(A, -B)
branch_equal = A - B

fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.25), constrained_layout=True)
for ax, field, title in [
    (axes[0], union, "union: max(A, B)"),
    (axes[1], intersection, "intersection: min(A, B)"),
    (axes[2], difference, "difference: min(A, -B)"),
]:
    ax.contourf(X, Y, field, levels=np.linspace(-1.0, 0.9, 18), cmap="RdYlBu_r", alpha=0.84)
    ax.contour(X, Y, field, levels=[0.0], colors="black", linewidths=2.2)
    ax.contour(X, Y, branch_equal, levels=[0.0], colors=PALETTE["gold"], linewidths=1.3, linestyles="dashed")
    style_axis(ax, title, equal=True, xlabel="x", ylabel="y")
csg_path = save_figure(fig, "csg-operators-and-creases.png")

validation_data["csg"] = {
    "union_contains_A": bool(np.all(union >= A - 1e-12)),
    "union_contains_B": bool(np.all(union >= B - 1e-12)),
    "intersection_below_A": bool(np.all(intersection <= A + 1e-12)),
    "intersection_below_B": bool(np.all(intersection <= B + 1e-12)),
    "difference_removed_sample": float(difference[np.abs(y).argmin(), np.abs(x - 0.55).argmin()]),
}
display_artifact(csg_path, width=900)
validation_data["csg"]


## 7. Warping by inverse sampling

A warp changes the field query, not the final mesh vertices. To render or polygonize a warped primitive, evaluate the base distance function at an inverse-warped point. That is why normals need care: the gradient should transform by the Jacobian of the sampling map, not by simply warping the old normal vector.

The panels below use a capsule field. Twist rotates the sample coordinates by an angle that depends on height. Taper rescales horizontal distance with height. Bend maps a straight capsule coordinate system into an arc. These are 2D slices of the same idea used by 3D Barr-style deformations.


In [ ]:
def capsule_level(points):
    d, _, _ = segment_distance_2d(points, np.array([-0.9, 0.0]), np.array([0.9, 0.0]))
    return 0.34 - d


def inverse_twist(points, strength=1.25):
    p = np.asarray(points, dtype=float)
    x = p[..., 0]
    y = p[..., 1]
    angle = -strength * y
    c = np.cos(angle)
    s = np.sin(angle)
    return np.stack([c * x - s * y, s * x + c * y], axis=-1)


def inverse_taper(points, amount=0.35):
    p = np.asarray(points, dtype=float)
    scale = np.maximum(0.35, 1.0 + amount * p[..., 1])
    return np.stack([p[..., 0] / scale, p[..., 1]], axis=-1)


def inverse_bend(points, radius=1.45):
    p = np.asarray(points, dtype=float)
    Xp = p[..., 0]
    Yp = p[..., 1]
    angle = np.arctan2(Xp, Yp + radius)
    u = radius * angle
    v = np.sqrt(Xp * Xp + (Yp + radius) ** 2) - radius
    return np.stack([u, v], axis=-1)


x, y, X, Y, P = xy_grid(xlim=(-1.8, 1.8), ylim=(-1.45, 1.45), n=320)
warp_cases = [
    ("base capsule", P),
    ("twist inverse sample", inverse_twist(P)),
    ("taper inverse sample", inverse_taper(P)),
    ("bend inverse sample", inverse_bend(P)),
]

fig, axes = plt.subplots(1, 4, figsize=(15.4, 3.9), constrained_layout=True)
for ax, (title, query_points) in zip(axes, warp_cases):
    field = capsule_level(query_points)
    ax.contourf(X, Y, field, levels=np.linspace(-0.85, 0.34, 18), cmap="cividis", alpha=0.82)
    ax.contour(X, Y, field, levels=[0.0], colors="white", linewidths=2.2)
    style_axis(ax, title, equal=True, xlabel="x", ylabel="y")
warp_path = save_figure(fig, "warp-inverse-sampling.png")

probe = np.array([[0.45, 0.4], [1.1, -0.25], [-0.75, 0.55]])
validation_data["warping"] = {
    "base_values_at_probe": [float(v) for v in capsule_level(probe)],
    "twisted_values_at_probe": [float(v) for v in capsule_level(inverse_twist(probe))],
    "tapered_values_at_probe": [float(v) for v in capsule_level(inverse_taper(probe))],
    "bent_values_at_probe": [float(v) for v in capsule_level(inverse_bend(probe))],
}
display_artifact(warp_path, width=920)
validation_data["warping"]


## 8. BlobTrees as expression graphs

A BlobTree is an evaluation tree for implicit fields. Leaves are primitives. Interior nodes are operators: affine transforms, warps, blends, booleans, and material-like annotations in richer systems. Rendering only needs a field value and, usually, a gradient at the query point; the tree describes how to obtain those values without flattening the model into one formula by hand.

The graph below is intentionally small. It combines two skeletal blobs with a Ricci blend, twists the result, subtracts a cutter, and unions that with a support cylinder-like field. The labels show the operation that would be applied during a bottom-up traversal.


In [ ]:
G = nx.DiGraph()
nodes = {
    "root": "union max",
    "diff": "difference",
    "support": "line skeleton",
    "twist": "warp: twist",
    "blend": "Ricci blend n=3",
    "left": "point blob A",
    "right": "point blob B",
    "cutter": "box cutter",
}
for node, label in nodes.items():
    G.add_node(node, label=label)
G.add_edges_from([
    ("root", "diff"),
    ("root", "support"),
    ("diff", "twist"),
    ("diff", "cutter"),
    ("twist", "blend"),
    ("blend", "left"),
    ("blend", "right"),
])

pos = {
    "root": (0.0, 3.1),
    "diff": (-1.35, 2.2),
    "support": (1.35, 2.2),
    "twist": (-2.0, 1.25),
    "cutter": (-0.75, 1.25),
    "blend": (-2.0, 0.3),
    "left": (-2.55, -0.65),
    "right": (-1.45, -0.65),
}

fig, ax = plt.subplots(figsize=(9.2, 6.1))
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=18, edge_color=PALETTE["gray"], width=1.8)
for node, (xx, yy) in pos.items():
    color = PALETTE["blue"] if G.out_degree(node) else PALETTE["teal"]
    box = patches.FancyBboxPatch((xx - 0.55, yy - 0.18), 1.1, 0.36, boxstyle="round,pad=0.08", facecolor=color, edgecolor="white", linewidth=1.8)
    ax.add_patch(box)
    ax.text(xx, yy, G.nodes[node]["label"], ha="center", va="center", fontsize=9, color="white")
ax.set_xlim(-3.1, 2.15)
ax.set_ylim(-1.05, 3.55)
ax.set_axis_off()
ax.set_title("BlobTree traversal composes field operations", fontsize=12, color=PALETTE["ink"])
blobtree_path = save_figure(fig, "blobtree-evaluation-graph.png")

postorder = list(nx.dfs_postorder_nodes(G, source="root"))
validation_data["blobtree"] = {
    "node_count": G.number_of_nodes(),
    "edge_count": G.number_of_edges(),
    "is_tree": bool(nx.is_arborescence(G)),
    "postorder": postorder,
    "root_evaluated_last": bool(postorder[-1] == "root"),
}
display_artifact(blobtree_path, width=760)
validation_data["blobtree"]


## 9. Marching-cubes extraction from a synthetic field

Polygonization turns a sampled implicit field into a mesh for conventional rendering. This is not the only way to render an implicit surface, but it is the route most compatible with ordinary mesh pipelines. The extraction below samples a small 3D field made from two point primitives and one line-segment primitive, then runs marching cubes at a fixed iso value.

The generated HTML artifact is interactive. Rotate it and look for two things: the merged neck where the fields blend, and the faceting pattern left by sampling on a finite grid. The diagnostics record vertex count, face count, Euler characteristic, bounding box, and normal length statistics so a later edit can catch blank fields or broken extraction.


In [ ]:
def synthetic_implicit_volume(n=54, extent=1.75):
    axis = np.linspace(-extent, extent, n)
    X, Y, Z = np.meshgrid(axis, axis, axis, indexing="ij")
    P3 = np.stack([X, Y, Z], axis=-1)
    c0 = np.array([-0.55, -0.08, 0.0])
    c1 = np.array([0.58, 0.05, 0.02])
    d0 = np.linalg.norm(P3 - c0, axis=-1)
    d1 = np.linalg.norm(P3 - c1, axis=-1)
    dseg, _, _ = segment_distance_3d(P3, np.array([-0.85, 0.0, -0.55]), np.array([0.85, 0.0, 0.55]))
    volume = compact_falloff(d0, support=0.95) + compact_falloff(d1, support=0.95) + 0.72 * compact_falloff(dseg, support=0.48)
    return axis, volume


axis, volume = synthetic_implicit_volume()
level = 0.42
spacing = (float(axis[1] - axis[0]),) * 3
verts, faces, normals, values = measure.marching_cubes(volume, level=level, spacing=spacing)
verts = verts + axis[0]
mesh = trimesh.Trimesh(vertices=verts, faces=faces, process=False)

fig = go.Figure(
    data=[
        go.Mesh3d(
            x=verts[:, 0],
            y=verts[:, 1],
            z=verts[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            intensity=verts[:, 2],
            colorscale="Viridis",
            opacity=1.0,
            flatshading=False,
            name="F = iso",
        )
    ]
)
fig.update_layout(
    title="Marching cubes on a blended skeletal field",
    scene=dict(
        xaxis_title="x",
        yaxis_title="y",
        zaxis_title="z",
        aspectmode="data",
        camera=dict(eye=dict(x=1.8, y=1.65, z=1.15)),
    ),
    margin=dict(l=0, r=0, t=44, b=0),
)
mesh_html_path = record_artifact(save_plotly_html(fig, TOPIC, "marching-cubes-synthetic-implicit-field.html", include_plotlyjs="cdn"))

normal_lengths = np.linalg.norm(normals, axis=1)
mesh_diagnostics = {
    "iso_level": level,
    "grid_shape": list(volume.shape),
    "volume_min_max": [float(volume.min()), float(volume.max())],
    "vertex_count": int(len(verts)),
    "face_count": int(len(faces)),
    "euler_number": int(mesh.euler_number),
    "watertight": bool(mesh.is_watertight),
    "bounds_min": [float(v) for v in mesh.bounds[0]],
    "bounds_max": [float(v) for v in mesh.bounds[1]],
    "normal_length_mean": float(normal_lengths.mean()),
    "normal_length_max_error": float(np.max(np.abs(normal_lengths - 1.0))),
}
mesh_check_path = save_check(mesh_diagnostics, "marching-cubes-mesh-diagnostics.json")
validation_data["marching_cubes"] = mesh_diagnostics

display_artifact(mesh_html_path, width="100%", height=560)
display_artifact(mesh_check_path)
mesh_diagnostics


## Applied lab: design an implicit operator stack

Use the small fields above as a lab bench for an implicit-modeling design. The task is to create a field expression for a handle-like object without editing a vertex mesh:

1. Start with a line-segment skeletal primitive for the grip.
2. Add two point primitives for rounded end caps.
3. Use a blend to make the caps attach smoothly.
4. Subtract a box or circle field to cut a notch.
5. Apply a bend warp to curve the grip.
6. Extract the result on a grid and check the mesh has nonzero faces.

The important design habit is to record the operator tree before tuning constants. If a later result is too bulgy, too sharp, or too slow, the tree tells you whether to change a falloff support, a blend exponent, a CSG branch, a warp, or the grid resolution.


In [ ]:
probe_points = np.array([
    [-0.75, 0.0],
    [0.0, 0.0],
    [0.75, 0.0],
    [0.0, 0.42],
    [0.0, -0.42],
])


def handle_grip_field(points):
    d, _, _ = segment_distance_2d(points, np.array([-0.95, 0.0]), np.array([0.95, 0.0]))
    return compact_falloff(d, support=0.48)


def handle_caps_field(points):
    left = compact_falloff(np.linalg.norm(points - np.array([-0.95, 0.0]), axis=-1), support=0.58)
    right = compact_falloff(np.linalg.norm(points - np.array([0.95, 0.0]), axis=-1), support=0.58)
    return left + right


def notch_field(points):
    return positive_box_field(points, center=(0.0, 0.28), half=(0.32, 0.22))


straight_handle = handle_grip_field(probe_points) + handle_caps_field(probe_points)
notched_handle = np.minimum(straight_handle, -notch_field(probe_points))
bent_handle = handle_grip_field(inverse_bend(probe_points, radius=1.8)) + handle_caps_field(inverse_bend(probe_points, radius=1.8))

lab_rows = []
for point, straight, notched, bent in zip(probe_points, straight_handle, notched_handle, bent_handle):
    lab_rows.append({
        "x": float(point[0]),
        "y": float(point[1]),
        "straight_field": float(straight),
        "notched_field": float(notched),
        "bent_field": float(bent),
    })
lab_table_path = record_check(save_table_csv(lab_rows, TOPIC, "operator-stack-probe-values.csv"))
validation_data["applied_lab"] = {
    "probe_count": len(lab_rows),
    "straight_center_value": float(straight_handle[1]),
    "notch_reduces_upper_probe": bool(notched_handle[3] < straight_handle[3]),
    "bent_values_finite": bool(np.all(np.isfinite(bent_handle))),
}
display_artifact(lab_table_path)
validation_data["applied_lab"]


## Sanity checks

The checks below are intentionally redundant. They verify the generated artifacts, make sure the key images are nonblank, and assert the identities that the lesson depends on: additive blending dominates each operand, hard union equals `max`, CSG union/intersection have the expected ordering, marching-cubes output is nonempty, and BlobTree traversal evaluates the root last.


In [ ]:
image_paths = [path for path in generated_artifacts if path.suffix.lower() == ".png"]
html_paths = [path for path in generated_artifacts if path.suffix.lower() == ".html"]
all_artifact_records = assert_artifacts([*generated_artifacts, *check_artifacts], min_bytes=1200)
image_records = [assert_nonblank_image(path) for path in image_paths]

assert validation_data["scalar_field"]["boundary_max_abs"] < 1e-12
assert validation_data["blending"]["sum_greater_or_equal_operands"]
assert validation_data["blending"]["hard_union_equals_max"]
assert validation_data["csg"]["union_contains_A"]
assert validation_data["csg"]["union_contains_B"]
assert validation_data["csg"]["intersection_below_A"]
assert validation_data["csg"]["intersection_below_B"]
assert validation_data["partitioning"]["active_cell_count"] > 0
assert validation_data["partitioning"]["marching_segment_count"] > 0
assert validation_data["blobtree"]["is_tree"]
assert validation_data["blobtree"]["root_evaluated_last"]
assert validation_data["marching_cubes"]["vertex_count"] > 100
assert validation_data["marching_cubes"]["face_count"] > 100
assert validation_data["marching_cubes"]["normal_length_max_error"] < 1e-5
assert validation_data["applied_lab"]["notch_reduces_upper_probe"]

final_report = {
    "chapter": CHAPTER,
    "title": TITLE,
    "printed_pages": PRINTED_PAGES,
    "pdf_pages": PDF_PAGES,
    "artifact_count": len(generated_artifacts),
    "check_artifact_count": len(check_artifacts),
    "image_count": len(image_paths),
    "html_count": len(html_paths),
    "concepts_checked": sorted(validation_data.keys()),
    "artifacts": [str(path.relative_to(BOOK_ROOT)).replace("\\", "/") for path in generated_artifacts],
    "checks": validation_data,
}
final_path = save_check(final_report, "final-sanity.json")
assert_artifacts([final_path], min_bytes=40)
display_artifact(final_path)
final_report


## Takeaways

Implicit modeling replaces vertex editing with field design. A primitive is a distance query plus a falloff. Blending, CSG, warping, and BlobTrees are ways to compose those queries before the final surface is found. This is why the same model can be ray traced, sampled into a grid, polygonized with marching cubes, or cached at tree nodes for interactive editing.

The main implementation risks are all visible in the artifacts: a wrong sign convention flips normals or booleans, an overly soft additive blend creates unintended bulges, a hard `min` or `max` operator creates a crease, a warp needs inverse sampling and a Jacobian-aware normal, and a coarse grid can miss or ambiguously connect surface pieces. The sanity checks keep those risks tied to measurable field, mesh, and graph data rather than to screenshots alone.
